# Rice-Only Yield Model Stabilization

Notebook ini melatih model khusus `Rice` agar target `Yield` tidak tercampur dengan skala komoditas lain. `Production` hanya dipakai untuk audit kualitas label, bukan sebagai fitur model, karena `Yield` berkaitan langsung dengan `Production / Area`.

In [1]:
import json
import re
import warnings
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42

try:
    from xgboost import XGBRegressor
except Exception as exc:
    XGBRegressor = None
    XGBOOST_IMPORT_ERROR = exc

try:
    from lightgbm import LGBMRegressor
except Exception as exc:
    LGBMRegressor = None
    LIGHTGBM_IMPORT_ERROR = exc

In [2]:
# Isi manual jika auto-detect salah.
# Contoh Kaggle input: '/kaggle/input/crop-yield-data/Crop Yeild Data (1).csv'
TRAIN_CSV_PATH = "/kaggle/input/datasets/brspot/dataset2/Crop Yeild Data.csv"

TARGET_COL = 'Yield'
RICE_CROP_NAME = 'Rice'
DEFAULT_SEASON_IF_MISSING = np.nan

# Filter khusus Rice. Baris yang dibuang tetap disimpan sebagai audit.
YIELD_MIN_FOR_TRAINING = 0.5
YIELD_MAX_FOR_TRAINING = 6.0
MAX_PROD_AREA_REL_DIFF_PCT = 50.0

# Fitur utama: tanpa Production dan tanpa State.
RICE_FEATURES = [
    'Crop_Year',
    'Season',
    'Area',
    'Annual_Rainfall',
    'Fertilizer',
    'Pesticide',
    'Avg_Temperature',
    'Max_Temperature',
    'Min_Temperature'
]

NUMERIC_COLS = [
    'Crop_Year', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide',
    'Avg_Temperature', 'Max_Temperature', 'Min_Temperature', TARGET_COL
]

N_SPLITS = 5
OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
MODEL_DIR = OUTPUT_DIR / 'rice_trained_models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
def clean_column_names(df):
    df = df.copy()
    df.columns = [re.sub(r'[^0-9a-zA-Z_]+', '_', str(c).strip()).strip('_') for c in df.columns]
    return df

COLUMN_ALIASES = {
    'Crop': ['Crop', 'Komoditas'],
    'Crop_Year': ['Crop_Year', 'Tahun', 'Year'],
    'Season': ['Season', 'Musim'],
    'State': ['State', 'Provinsi', 'Region'],
    'Area': ['Area', 'Luas_Panen'],
    'Production': ['Production', 'Produksi'],
    'Annual_Rainfall': ['Annual_Rainfall', 'Rainfall_mm'],
    'Fertilizer': ['Fertilizer', 'Fertilizer_Used_kg'],
    'Pesticide': ['Pesticide', 'Pesticides_Used_kg'],
    'Avg_Temperature': ['Avg_Temperature', 'Temperature_C'],
    'Max_Temperature': ['Max_Temperature'],
    'Min_Temperature': ['Min_Temperature'],
    'Yield': ['Yield', 'Produktivitas']
}

CROP_NAME_MAP = {
    'padi': 'Rice',
    'padi sawah': 'Rice',
    'padi ladang': 'Rice',
    'beras': 'Rice',
    'rice': 'Rice'
}

def standardize_crop_yield_schema(df):
    df = clean_column_names(df)
    for target_col, aliases in COLUMN_ALIASES.items():
        if target_col in df.columns:
            continue
        for alias in aliases:
            clean_alias = re.sub(r'[^0-9a-zA-Z_]+', '_', str(alias).strip()).strip('_')
            if clean_alias in df.columns:
                df[target_col] = df[clean_alias]
                break

    if 'Crop' in df.columns:
        df['Crop'] = df['Crop'].astype(str).str.strip().apply(lambda value: CROP_NAME_MAP.get(value.lower(), value))
    if 'Season' not in df.columns:
        df['Season'] = DEFAULT_SEASON_IF_MISSING
    if 'Max_Temperature' not in df.columns:
        df['Max_Temperature'] = np.nan
    if 'Min_Temperature' not in df.columns:
        df['Min_Temperature'] = np.nan
    return df

def search_files(pattern):
    roots = [Path('/kaggle/input'), Path('/kaggle/working'), Path('.'), Path.home() / 'Downloads']
    found = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for path in sorted(root.rglob(pattern)):
            if not path.is_file():
                continue
            key = str(path.resolve())
            if key not in seen:
                seen.add(key)
                found.append(path)
    return found

def choose_train_csv():
    if TRAIN_CSV_PATH:
        path = Path(TRAIN_CSV_PATH)
        if not path.exists():
            raise FileNotFoundError(f'TRAIN_CSV_PATH tidak ditemukan: {path}')
        return path

    required = {'Crop', 'Crop_Year', 'Season', 'Area', 'Production', 'Annual_Rainfall', 'Fertilizer', 'Pesticide', TARGET_COL}
    candidates = []
    for path in search_files('*.csv'):
        try:
            preview = standardize_crop_yield_schema(pd.read_csv(path, nrows=5))
            cols = set(preview.columns)
            score = len(required & cols)
            if RICE_CROP_NAME.lower() in ''.join(preview.get('Crop', pd.Series(dtype=str)).astype(str).str.lower().tolist()):
                score += 3
            if any(word in str(path).lower() for word in ['bps', 'indonesia', 'template']):
                score -= 4
            if score >= 7:
                candidates.append((score, path, list(preview.columns)))
        except Exception as exc:
            print('Lewati CSV:', path, exc)

    if not candidates:
        available = '\n'.join(str(p) for p in search_files('*.csv')[:30]) or '(tidak ada file CSV ditemukan)'
        raise FileNotFoundError(f'Dataset training tidak terdeteksi. Isi TRAIN_CSV_PATH manual.\nCSV ditemukan:\n{available}')

    candidates = sorted(candidates, key=lambda item: item[0], reverse=True)
    print('[OK] Dataset training terpilih:', candidates[0][1])
    print('Kolom:', candidates[0][2])
    return candidates[0][1]

def percent_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.where(y_true != 0, np.abs(y_true - y_pred) / np.abs(y_true) * 100, np.nan)

def metric_row(model_name, y_true, y_pred):
    ape = percent_error(y_true, y_pred)
    return {
        'model': model_name,
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else np.nan,
        'MAPE': float(np.nanmean(ape)),
        'median_percent_error': float(np.nanmedian(ape)),
        'p90_percent_error': float(np.nanpercentile(ape, 90)),
        'p95_percent_error': float(np.nanpercentile(ape, 95)),
        'max_percent_error': float(np.nanmax(ape)),
        'pct_predictions_error_le_15': float(np.nanmean(ape <= 15) * 100),
        'n': int(len(y_true))
    }

def build_pipeline(X, regressor):
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
    preprocessor = ColumnTransformer([
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
    ], remainder='drop')
    return Pipeline([('preprocess', preprocessor), ('model', regressor)])

def make_log_target(regressor):
    return TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)

In [4]:
train_csv_path = choose_train_csv()
raw_df = standardize_crop_yield_schema(pd.read_csv(train_csv_path))

required_cols = sorted(set(['Crop', 'Production', TARGET_COL] + RICE_FEATURES))
missing_cols = [col for col in required_cols if col not in raw_df.columns]
if missing_cols:
    raise ValueError(f'Kolom wajib tidak ditemukan: {missing_cols}')

for col in NUMERIC_COLS:
    if col in raw_df.columns:
        raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')

raw_df = raw_df.drop_duplicates().reset_index(drop=True)
rice_raw = raw_df[raw_df['Crop'].astype(str).str.strip().str.lower().eq(RICE_CROP_NAME.lower())].copy()
rice_raw = rice_raw.reset_index(drop=False).rename(columns={'index': 'original_row_index'})

if rice_raw.empty:
    raise ValueError(f'Tidak ada baris Crop == {RICE_CROP_NAME!r}.')

rice_raw['yield_from_production_area'] = np.where(rice_raw['Area'] > 0, rice_raw['Production'] / rice_raw['Area'], np.nan)
rice_raw['yield_prod_area_rel_diff_pct'] = np.where(
    rice_raw[TARGET_COL] != 0,
    np.abs(rice_raw[TARGET_COL] - rice_raw['yield_from_production_area']) / np.abs(rice_raw[TARGET_COL]) * 100,
    np.nan
)

def removal_reason(row):
    reasons = []
    if pd.isna(row[TARGET_COL]) or row[TARGET_COL] <= 0:
        reasons.append('Yield <= 0 atau missing')
    if pd.notna(row[TARGET_COL]) and row[TARGET_COL] < YIELD_MIN_FOR_TRAINING:
        reasons.append(f'Yield < {YIELD_MIN_FOR_TRAINING}')
    if pd.notna(row[TARGET_COL]) and row[TARGET_COL] > YIELD_MAX_FOR_TRAINING:
        reasons.append(f'Yield > {YIELD_MAX_FOR_TRAINING}')
    if pd.notna(row['yield_prod_area_rel_diff_pct']) and row['yield_prod_area_rel_diff_pct'] > MAX_PROD_AREA_REL_DIFF_PCT:
        reasons.append(f'|Yield - Production/Area| > {MAX_PROD_AREA_REL_DIFF_PCT}%')
    return '; '.join(reasons)

rice_raw['removal_reason'] = rice_raw.apply(removal_reason, axis=1)
removed_rice_df = rice_raw[rice_raw['removal_reason'] != ''].copy()
rice_clean_df = rice_raw[rice_raw['removal_reason'] == ''].copy().reset_index(drop=True)

if len(rice_clean_df) < 50:
    raise ValueError(f'Data Rice bersih terlalu sedikit: {len(rice_clean_df)} baris.')

clean_path = OUTPUT_DIR / 'rice_training_cleaned.csv'
model_ready_path = OUTPUT_DIR / 'rice_training_model_ready.csv'
audit_path = OUTPUT_DIR / 'rice_removed_rows_audit.csv'
rice_clean_df.to_csv(clean_path, index=False)
rice_clean_df.to_csv(model_ready_path, index=False)
removed_rice_df.to_csv(audit_path, index=False)

print('[OK] Rice raw rows:', len(rice_raw))
print('[OK] Rice cleaned rows:', len(rice_clean_df))
print('[OK] Removed/audit rows:', len(removed_rice_df))
print('[OK] Cleaned Rice disimpan:', clean_path)
print('[OK] Model-ready Rice disimpan:', model_ready_path)
print('[OK] Audit row dibuang disimpan:', audit_path)

display(rice_clean_df[TARGET_COL].describe(percentiles=[.05, .1, .25, .5, .75, .9, .95]).to_frame('clean_rice_yield'))
display(removed_rice_df[['original_row_index', 'Crop_Year', 'Season', 'State', 'Area', 'Production', TARGET_COL, 'yield_from_production_area', 'yield_prod_area_rel_diff_pct', 'removal_reason']].head(30))

[OK] Rice raw rows: 1197
[OK] Rice cleaned rows: 1168
[OK] Removed/audit rows: 29
[OK] Cleaned Rice disimpan: /kaggle/working/rice_training_cleaned.csv
[OK] Model-ready Rice disimpan: /kaggle/working/rice_training_model_ready.csv
[OK] Audit row dibuang disimpan: /kaggle/working/rice_removed_rows_audit.csv


,clean_rice_yield
count,1168.000000
mean,2.213864
std,0.765473
min,0.515000
5%,1.024700
10%,1.211700
25%,1.632500
50%,2.207500
75%,2.731500
90%,3.187100


,original_row_index,Crop_Year,Season,State,Area,Production,Yield,yield_from_production_area,yield_prod_area_rel_diff_pct,removal_reason
37,409,1999,Rabi,Meghalaya,3929.00,13846,1.756,3.524052,100.686328,|Yield - Production/Area| > 50.0%
56,678,2000,Rabi,Meghalaya,4003.00,13135,1.744,3.281289,88.147307,|Yield - Production/Area| > 50.0%
119,1620,2004,Summer,Meghalaya,9224.00,37439,2.078,4.058868,95.325706,|Yield - Production/Area| > 50.0%
134,1855,2005,Summer,Meghalaya,9791.00,39910,2.245,4.076192,81.567591,|Yield - Production/Area| > 50.0%
144,1982,2006,Rabi,Meghalaya,9854.00,39940,2.077,4.053176,95.145709,|Yield - Production/Area| > 50.0%
192,2705,2008,Summer,Meghalaya,12752.00,46600,2.154,3.654329,69.653145,|Yield - Production/Area| > 50.0%
213,3035,2009,Summer,Meghalaya,12777.00,46943,2.171,3.674024,69.231858,|Yield - Production/Area| > 50.0%
231,3307,2010,Rabi,Meghalaya,12797.00,47413,2.077,3.705009,78.382715,|Yield - Production/Area| > 50.0%
248,3552,2011,Rabi,Meghalaya,12836.00,54206,2.070,4.222967,104.008051,|Yield - Production/Area| > 50.0%
263,3815,2012,Rabi,Meghalaya,13267.00,56786,2.130,4.280244,100.950433,|Yield - Production/Area| > 50.0%


In [5]:
available_features = [col for col in RICE_FEATURES if col in rice_clean_df.columns]
X = rice_clean_df[available_features].copy()
y = rice_clean_df[TARGET_COL].astype(float).copy()

candidate_models = {
    'BaselineMean': make_log_target(DummyRegressor(strategy='mean')),
    'LinearRegression': make_log_target(LinearRegression()),
    'Ridge': make_log_target(Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    'Lasso': make_log_target(Lasso(alpha=0.001, random_state=RANDOM_STATE, max_iter=10000)),
    'ElasticNet': make_log_target(ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=10000)),
    'DecisionTree': make_log_target(DecisionTreeRegressor(max_depth=8, min_samples_leaf=2, random_state=RANDOM_STATE)),
    'RandomForest': make_log_target(RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)),
    'ExtraTrees': make_log_target(ExtraTreesRegressor(n_estimators=300, max_depth=12, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)),
    'GradientBoosting': make_log_target(GradientBoostingRegressor(random_state=RANDOM_STATE)),
    'AdaBoost': make_log_target(AdaBoostRegressor(n_estimators=200, random_state=RANDOM_STATE)),
    'KNN': make_log_target(KNeighborsRegressor(n_neighbors=7)),
    'SVR': make_log_target(SVR(kernel='rbf', C=10, epsilon=0.05)),
}

skipped_optional_models = []
if XGBRegressor is not None:
    candidate_models['XGBoost'] = make_log_target(XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='reg:squarederror',
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
else:
    skipped_optional_models.append({'model': 'XGBoost', 'reason': str(XGBOOST_IMPORT_ERROR)})

if LGBMRegressor is not None:
    candidate_models['LightGBM'] = make_log_target(LGBMRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ))
else:
    skipped_optional_models.append({'model': 'LightGBM', 'reason': str(LIGHTGBM_IMPORT_ERROR)})

print('Model kandidat:', list(candidate_models.keys()))
if skipped_optional_models:
    print('Model optional dilewati:')
    display(pd.DataFrame(skipped_optional_models))

cv = KFold(n_splits=min(N_SPLITS, len(rice_clean_df)), shuffle=True, random_state=RANDOM_STATE)
metric_rows = []
prediction_frames = []
successful_candidate_models = {}
failed_model_rows = list(skipped_optional_models)

for model_name, regressor in candidate_models.items():
    print('Validasi model:', model_name)
    pipe = build_pipeline(X, regressor)
    try:
        cv_pred = cross_val_predict(pipe, X, y, cv=cv)
    except Exception as exc:
        failed_model_rows.append({'model': model_name, 'stage': 'cross_validation', 'reason': str(exc)})
        print('[SKIP]', model_name, exc)
        continue

    metric_rows.append(metric_row(model_name, y, cv_pred))
    successful_candidate_models[model_name] = regressor

    pred_df = rice_clean_df[['original_row_index', 'Crop_Year', 'Season', 'State', 'Area', 'Production', TARGET_COL]].copy()
    pred_df['model'] = model_name
    pred_df['predicted_yield'] = cv_pred
    pred_df['absolute_error'] = np.abs(pred_df[TARGET_COL] - pred_df['predicted_yield'])
    pred_df['percent_error'] = percent_error(pred_df[TARGET_COL], pred_df['predicted_yield'])
    prediction_frames.append(pred_df)

if not metric_rows:
    raise RuntimeError('Tidak ada model yang berhasil divalidasi.')

rice_cv_results_df = pd.DataFrame(metric_rows).sort_values(['p90_percent_error', 'median_percent_error', 'MAE']).reset_index(drop=True)
rice_cv_predictions_df = pd.concat(prediction_frames, ignore_index=True)

results_path = OUTPUT_DIR / 'rice_model_cv_results.csv'
predictions_path = OUTPUT_DIR / 'rice_model_cv_predictions.csv'
failed_models_path = OUTPUT_DIR / 'rice_model_failed_or_skipped.csv'
rice_cv_results_df.to_csv(results_path, index=False)
rice_cv_predictions_df.to_csv(predictions_path, index=False)
pd.DataFrame(failed_model_rows).to_csv(failed_models_path, index=False)

print('[OK] Hasil validasi disimpan:', results_path)
print('[OK] Prediksi validasi disimpan:', predictions_path)
print('[OK] Model gagal/optional dilewati disimpan:', failed_models_path)
display(rice_cv_results_df)
if failed_model_rows:
    display(pd.DataFrame(failed_model_rows))

Model kandidat: ['BaselineMean', 'LinearRegression', 'Ridge', 'Lasso', 'ElasticNet', 'DecisionTree', 'RandomForest', 'ExtraTrees', 'GradientBoosting', 'AdaBoost', 'KNN', 'SVR', 'XGBoost', 'LightGBM']
Validasi model: BaselineMean
Validasi model: LinearRegression
Validasi model: Ridge
Validasi model: Lasso
Validasi model: ElasticNet
Validasi model: DecisionTree
Validasi model: RandomForest
Validasi model: ExtraTrees
Validasi model: GradientBoosting
Validasi model: AdaBoost
Validasi model: KNN
Validasi model: SVR
Validasi model: XGBoost
Validasi model: LightGBM
[OK] Hasil validasi disimpan: /kaggle/working/rice_model_cv_results.csv
[OK] Prediksi validasi disimpan: /kaggle/working/rice_model_cv_predictions.csv
[OK] Model gagal/optional dilewati disimpan: /kaggle/working/rice_model_failed_or_skipped.csv


,model,MAE,RMSE,R2,MAPE,median_percent_error,p90_percent_error,p95_percent_error,max_percent_error,pct_predictions_error_le_15,n
0,XGBoost,0.262055,0.370362,0.765704,13.625238,8.811459,28.647223,39.703417,166.868143,70.291096,1168
1,LightGBM,0.264915,0.373705,0.761455,13.639437,8.536981,29.430905,39.262879,151.808587,69.606164,1168
2,RandomForest,0.270900,0.386454,0.744901,14.199083,9.083622,29.488610,44.549537,194.134741,70.291096,1168
3,ExtraTrees,0.262126,0.372531,0.762951,13.787855,8.593544,29.932583,41.330318,185.348842,71.404110,1168
4,SVR,0.298756,0.416628,0.703510,15.820169,9.995550,33.318625,49.927281,212.565180,66.695205,1168
5,GradientBoosting,0.320331,0.433899,0.678419,17.071552,11.430298,34.915434,49.909615,195.272738,60.359589,1168
6,KNN,0.317729,0.442827,0.665049,16.710136,11.065864,35.990011,53.490458,206.832393,62.071918,1168
7,AdaBoost,0.421319,0.530383,0.519501,20.923563,16.933990,39.508797,50.527200,159.592264,45.119863,1168
8,DecisionTree,0.345199,0.497347,0.577494,18.700753,10.855264,41.765029,58.177231,299.927508,61.900685,1168
9,ElasticNet,0.493490,0.631846,0.318076,26.530376,18.600038,53.950684,72.798843,230.981868,40.239726,1168


In [6]:
best_model_path = MODEL_DIR / 'rice_yield_best_model.joblib'
manifest_rows = []
saved_model_paths = []
saved_pipelines = {}

for _, result_row in rice_cv_results_df.iterrows():
    model_name = result_row['model']
    regressor = successful_candidate_models[model_name]
    model_pipeline = build_pipeline(X, regressor)
    model_path = MODEL_DIR / f'rice_yield_{model_name}.joblib'

    print('Fit final dan simpan model:', model_name)
    try:
        model_pipeline.fit(X, y)
        joblib.dump(model_pipeline, model_path)
    except Exception as exc:
        failed_model_rows.append({'model': model_name, 'stage': 'final_fit_or_save', 'reason': str(exc)})
        print('[SKIP FINAL]', model_name, exc)
        continue

    saved_pipelines[model_name] = model_pipeline
    saved_model_paths.append(model_path)
    row = result_row.to_dict()
    row['model_name'] = model_name
    row['model_path'] = str(model_path)
    manifest_rows.append(row)

if not manifest_rows:
    raise RuntimeError('Tidak ada model final yang berhasil disimpan.')

best_model_name = manifest_rows[0]['model_name']
best_pipeline = saved_pipelines[best_model_name]
joblib.dump(best_pipeline, best_model_path)

model_manifest_df = pd.DataFrame(manifest_rows)
model_manifest_df['is_best'] = model_manifest_df['model_name'].eq(best_model_name)
manifest_order = ['model_name', 'model_path', 'is_best'] + [col for col in model_manifest_df.columns if col not in ['model_name', 'model_path', 'is_best']]
model_manifest_df = model_manifest_df[manifest_order]

manifest_path = OUTPUT_DIR / 'rice_model_manifest.csv'
model_manifest_df.to_csv(manifest_path, index=False)
pd.DataFrame(failed_model_rows).to_csv(failed_models_path, index=False)

zip_path = OUTPUT_DIR / 'rice_trained_models_all.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for model_path in saved_model_paths + [best_model_path]:
        zf.write(model_path, arcname=model_path.name)
    zf.write(manifest_path, arcname=manifest_path.name)
    zf.write(results_path, arcname=results_path.name)
    zf.write(model_ready_path, arcname=model_ready_path.name)

best_cv_metrics = json.loads(model_manifest_df[model_manifest_df['is_best']].to_json(orient='records'))[0]
metadata = {
    'model_name': best_model_name,
    'best_model_path': str(best_model_path),
    'all_models_dir': str(MODEL_DIR),
    'models_zip_path': str(zip_path),
    'model_manifest_path': str(manifest_path),
    'model_ready_csv_path': str(model_ready_path),
    'training_csv_path': str(train_csv_path),
    'target_col': TARGET_COL,
    'features': available_features,
    'rice_raw_rows': int(len(rice_raw)),
    'rice_clean_rows': int(len(rice_clean_df)),
    'rice_removed_rows': int(len(removed_rice_df)),
    'filters': {
        'crop': RICE_CROP_NAME,
        'yield_gt': 0,
        'yield_min_for_training': YIELD_MIN_FOR_TRAINING,
        'yield_max_for_training': YIELD_MAX_FOR_TRAINING,
        'max_prod_area_rel_diff_pct': MAX_PROD_AREA_REL_DIFF_PCT
    },
    'selection_metric': ['p90_percent_error', 'median_percent_error', 'MAE'],
    'best_cv_metrics': best_cv_metrics
}

metadata_path = OUTPUT_DIR / 'rice_yield_model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('[OK] Best model:', best_model_name)
print('[OK] Best model alias disimpan:', best_model_path)
print('[OK] Semua model disimpan di:', MODEL_DIR)
print('[OK] Manifest model disimpan:', manifest_path)
print('[OK] ZIP semua model disimpan:', zip_path)
print('[OK] Metadata disimpan:', metadata_path)
display(model_manifest_df)

Fit final dan simpan model: XGBoost
Fit final dan simpan model: LightGBM
Fit final dan simpan model: RandomForest
Fit final dan simpan model: ExtraTrees
Fit final dan simpan model: SVR
Fit final dan simpan model: GradientBoosting
Fit final dan simpan model: KNN
Fit final dan simpan model: AdaBoost
Fit final dan simpan model: DecisionTree
Fit final dan simpan model: ElasticNet
Fit final dan simpan model: Lasso
Fit final dan simpan model: Ridge
Fit final dan simpan model: LinearRegression
Fit final dan simpan model: BaselineMean
[OK] Best model: XGBoost
[OK] Best model alias disimpan: /kaggle/working/rice_trained_models/rice_yield_best_model.joblib
[OK] Semua model disimpan di: /kaggle/working/rice_trained_models
[OK] Manifest model disimpan: /kaggle/working/rice_model_manifest.csv
[OK] ZIP semua model disimpan: /kaggle/working/rice_trained_models_all.zip
[OK] Metadata disimpan: /kaggle/working/rice_yield_model_metadata.json


,model_name,model_path,is_best,model,MAE,RMSE,R2,MAPE,median_percent_error,p90_percent_error,p95_percent_error,max_percent_error,pct_predictions_error_le_15,n
0,XGBoost,/kaggle/working/rice_trained_models/rice_yield...,True,XGBoost,0.262055,0.370362,0.765704,13.625238,8.811459,28.647223,39.703417,166.868143,70.291096,1168
1,LightGBM,/kaggle/working/rice_trained_models/rice_yield...,False,LightGBM,0.264915,0.373705,0.761455,13.639437,8.536981,29.430905,39.262879,151.808587,69.606164,1168
2,RandomForest,/kaggle/working/rice_trained_models/rice_yield...,False,RandomForest,0.270900,0.386454,0.744901,14.199083,9.083622,29.488610,44.549537,194.134741,70.291096,1168
3,ExtraTrees,/kaggle/working/rice_trained_models/rice_yield...,False,ExtraTrees,0.262126,0.372531,0.762951,13.787855,8.593544,29.932583,41.330318,185.348842,71.404110,1168
4,SVR,/kaggle/working/rice_trained_models/rice_yield...,False,SVR,0.298756,0.416628,0.703510,15.820169,9.995550,33.318625,49.927281,212.565180,66.695205,1168
5,GradientBoosting,/kaggle/working/rice_trained_models/rice_yield...,False,GradientBoosting,0.320331,0.433899,0.678419,17.071552,11.430298,34.915434,49.909615,195.272738,60.359589,1168
6,KNN,/kaggle/working/rice_trained_models/rice_yield...,False,KNN,0.317729,0.442827,0.665049,16.710136,11.065864,35.990011,53.490458,206.832393,62.071918,1168
7,AdaBoost,/kaggle/working/rice_trained_models/rice_yield...,False,AdaBoost,0.421319,0.530383,0.519501,20.923563,16.933990,39.508797,50.527200,159.592264,45.119863,1168
8,DecisionTree,/kaggle/working/rice_trained_models/rice_yield...,False,DecisionTree,0.345199,0.497347,0.577494,18.700753,10.855264,41.765029,58.177231,299.927508,61.900685,1168
9,ElasticNet,/kaggle/working/rice_trained_models/rice_yield...,False,ElasticNet,0.493490,0.631846,0.318076,26.530376,18.600038,53.950684,72.798843,230.981868,40.239726,1168


In [12]:
# Sanity check: load ulang model tersimpan dan uji 1 baris Rice bersih.
# Ini hanya membuktikan file .joblib bisa dipanggil ulang, bukan ukuran performa utama.
SAMPLE_RICE_ROW_INDEX = 37

loaded_model = joblib.load(best_model_path)
sample_row = rice_clean_df.iloc[[SAMPLE_RICE_ROW_INDEX]].copy()
sample_X = sample_row[available_features]
sample_actual_y = float(sample_row[TARGET_COL].iloc[0])
sample_pred_y = float(loaded_model.predict(sample_X)[0])
sample_abs_error = abs(sample_actual_y - sample_pred_y)
sample_pct_error = sample_abs_error / sample_actual_y * 100 if sample_actual_y != 0 else np.nan

print('[OK] Model berhasil di-load ulang:', best_model_path)
print('Sample Rice row index:', SAMPLE_RICE_ROW_INDEX)
print('Actual Yield:', sample_actual_y)
print('Predicted Yield:', sample_pred_y)
print('Absolute Error:', sample_abs_error)
print('Percent Error:', sample_pct_error)

display(sample_row[available_features + [TARGET_COL]])
display(pd.DataFrame([{
    'model_name': best_model_name,
    'model_path': str(best_model_path),
    'sample_rice_row_index': SAMPLE_RICE_ROW_INDEX,
    'actual_yield': sample_actual_y,
    'predicted_yield': sample_pred_y,
    'absolute_error': sample_abs_error,
    'percent_error': sample_pct_error
}]))

[OK] Model berhasil di-load ulang: /kaggle/working/rice_trained_models/rice_yield_best_model.joblib
Sample Rice row index: 37
Actual Yield: 2.61
Predicted Yield: 2.6393251419067383
Absolute Error: 0.029325141906738406
Percent Error: 1.1235686554305904


,Crop_Year,Season,Area,Annual_Rainfall,Fertilizer,Pesticide,Avg_Temperature,Max_Temperature,Min_Temperature,Yield
37,1999,Kharif,25444.0,1434.588,2700371.72,6869.88,27.736,32.794,23.785,2.61


,model_name,model_path,sample_rice_row_index,actual_yield,predicted_yield,absolute_error,percent_error
0,XGBoost,/kaggle/working/rice_trained_models/rice_yield...,37,2.61,2.639325,0.029325,1.123569


In [11]:
import pandas as pd

file_path = "/kaggle/input/datasets/brspot/dataset2/Crop Yeild Data.csv"

df = pd.read_csv(file_path)

# Ambil baris ke-200 (ingat: index mulai dari 0)
row_200 = df.iloc[1]

print(row_200)

Crop                 Arhar/Tur
Crop_Year                 1997
Season             Kharif     
State                    Assam
Area                    6637.0
Production                4685
Annual_Rainfall         2051.4
Fertilizer           631643.29
Pesticide              2057.47
Yield                     0.71
Avg_Temperature         23.692
Max_Temperature         33.435
Min_Temperature         14.779
Name: 1, dtype: object
